# 与sft相比，rl之后，如何确定修改了哪些token的权重？

这是一个关于大模型训练后权重变化归因分析的深层技术问题。要确定RL后哪些token的权重被修改了，本质上是在问：
- <font color='red'>如何从参数空间的变化反推回token空间的贡献</font>。

可以通过以下多种方法组合实现：

## 核心思路：从参数差异到token归因
###  参数差异分析（最直接）
##### 首先比较SFT和RL后的模型权重：
- 关键观察：<font color='red'>RL训练通常只修改一小部分参数（尤其是靠近输出的层），而SFT往往改变更广泛的参数分布。</font>

In [ ]:
import torch
from safetensors import safe_open

# 加载两个模型的权重
sft_weights = {}
rl_weights = {}

# 逐层计算L2差异
for layer_name in sft_weights.keys():
    diff = torch.norm(rl_weights[layer_name] - sft_weights[layer_name], p=2)
    # 或者逐元素差异
    element_diff = (rl_weights[layer_name] - sft_weights[layer_name]).abs()

### <font color='blue'>梯度归因法（Gradient-based Attribution）</font>
##### <font color='red'>这是最接近"确定哪些token修改了权重"的方法：</font>
##### Integrated Gradients / 积分梯度


In [ ]:
from captum.attr import IntegratedGradients

# 对特定输入，计算每个token对最终输出的贡献
# 然后对比SFT和RL模型在相同输入上的归因差异

##### 具体步骤：
1. 选取一组测试prompt
2. 分别用SFT和RL模型生成回答
3. <font color='red'>对RL模型中概率变化最大的token，计算其梯度归因</font>
4. 反向追踪这些梯度流经哪些权重矩阵

### 逐层相关性传播（Layer-wise Relevance Propagation, LRP）

In [ ]:
# 对特定token的预测概率，逐层反向传播相关性分数
# 最终落在embedding层，即可知道哪些输入token"激活"了被修改的权重

## 基于训练动态的分析
###  梯度累积分析（训练时记录）
如果在RL训练过程中保存了梯度快照，可以直接分析：

In [ ]:
# RL训练中的PPO梯度
# ∇θ L_RLHF = -E[R(x) · ∇θ log πθ(x)]
# 
# 其中 ∇θ log πθ(x) 可以分解为每个token的梯度贡献

for token_idx in range(seq_len):
    token_grad = compute_token_gradient(token_idx, reward, log_prob)
    # 记录每个token对应的梯度范数
    token_grad_norms[token_idx] = torch.norm(token_grad)

##### 优势：
     这是最直接的方法，但需要修改训练代码来记录梯度。

### <font color='red'>Advantage加权分析</font>
<font color='red'>在PPO/RLHF中，每个token的梯度被Advantage函数 A($s_t,a_t$) 加权：</font>

∇θJ≈ t∑

##### 因此：
- Advantage值高的token → 对权重更新贡献大
  - <font color='red'>可以通过分析Advantage分布，确定"关键token"</font>

In [ ]:
# 从RL训练日志中提取
high_advantage_tokens = []
for step in training_logs:
    for token_idx, adv in enumerate(step.advantages):
        if abs(adv) > threshold:
            high_advantage_tokens.append({
                'token': tokenizer.decode([step.tokens[token_idx]]),
                'advantage': adv,
                'layer_grad_norm': step.grad_norms  # 如果有记录
            })

## 对比分析法
### 输出概率差异分析

In [ ]:
def analyze_token_prob_changes(sft_model, rl_model, prompts, tokenizer):
    results = []
    for prompt in prompts:
        sft_logits = sft_model(prompt).logits
        rl_logits = rl_model(prompt).logits
        
        # 计算每个位置的概率差异
        prob_diff = torch.softmax(rl_logits, -1) - torch.softmax(sft_logits, -1)
        
        # 找出差异最大的top-k token
        for pos in range(len(prompt)):
            top_changed = torch.topk(prob_diff[pos].abs(), k=5)
            results.append({
                'position': pos,
                'token': tokenizer.decode([top_changed.indices[0]]),
                'prob_change': top_changed.values[0].item()
            })
    return results

#### 逻辑：
- 如果某个token在RL后概率变化显著，说明其对应的权重（尤其是输出层和attention中的相关权重）被修改了。

### 神经元激活分析

In [ ]:
# 对比两个模型在相同输入上的神经元激活模式
# 使用t-SNE或UMAP可视化隐藏层表示

from sklearn.manifold import TSNE

# 提取某层的隐藏状态
sft_hidden = sft_model(input_ids, output_hidden_states=True).hidden_states[layer_idx]
rl_hidden = rl_model(input_ids, output_hidden_states=True).hidden_states[layer_idx]

# 对比差异
activation_diff = (rl_hidden - sft_hidden).norm(dim=-1)
# 激活差异大的位置 → 对应token的权重被修改

## 更高级的方法

### 权重分解（Weight Decomposition）
<font color='red'>对权重矩阵做SVD或低秩分解，分析RL后哪些奇异值/方向发生了变化：</font>


In [ ]:
# 对某层的W_q矩阵做SVD
U_sft, S_sft, Vh_sft = torch.linalg.svd(sft_Wq)
U_rl, S_rl, Vh_rl = torch.linalg.svd(rl_Wq)

# 比较奇异值变化
singular_diff = (S_rl - S_sft).abs()
# 变化大的奇异值对应特定的"功能方向"

### 概念激活向量（CAV, Concept Activation Vectors）
训练一个线性分类器来区分SFT和RL模型的隐藏表示，分析哪些概念/语义被改变。

## 实用建议

| 方法       | 难度  | 精度  | 适用场景           |
| -------- | --- | --- | -------------- |
| 参数差异L2范数 | ⭐   | 粗粒度 | 快速定位被修改的层      |
| <font color='red'>梯度归因（IG）</font> | ⭐⭐⭐ | 细粒度 | 分析特定输入的token贡献 |
| 训练时梯度记录  | ⭐⭐  | 最精确 | 需要修改训练代码       |
| 输出概率差异   | ⭐⭐  | 中等  | 无需修改训练代码       |
| 神经元激活分析  | ⭐⭐⭐ | 中等  | 可视化理解模型变化      |

## 总结
### 核心结论：
- 你无法直接说"token X的权重被修改了"，因为权重是共享的（所有token共用同一套参数）。但你可以通过以下间接方式确定：

1. 哪些token的生成概率/分布被RL显著改变 → 通过概率差异分析
2. 哪些token在RL训练中获得高Advantage/梯度 → 通过训练日志分析
3. 哪些权重矩阵/层被RL修改最多 → 通过参数差异分析
4. 特定输入中哪些token对输出变化贡献最大 → 通过梯度归因

这些方法组合使用，可以构建一个完整的"RL vs SFT权重变化归因"分析框架。